<a href="https://colab.research.google.com/github/pachir1su/file_swordmaster/blob/main/ALL_swordmaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install gradio pypdf pydub moviepy==1.0.3 reportlab mutagen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 5.2 MB/s eta 0:00:00


In [14]:
import gradio as gr
from pypdf import PdfReader, PdfWriter
from pydub import AudioSegment
from pydub.silence import split_on_silence
import os
import io
import zipfile
import mutagen

from moviepy.editor import VideoFileClip
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont

# 한글 폰트
pdfmetrics.registerFont(UnicodeCIDFont('HYGothic-Medium'))

# ==========================================
# 1. PDF 분할 (PDF Swordmaster)
# ==========================================
def parse_page_string(page_string, total_pages):
    pages = set()
    if not page_string: return []
    for part in page_string.split(','):
        part = part.strip()
        if '-' in part:
            start, end = map(int, part.split('-'))
            pages.update(range(start, end + 1))
        else:
            pages.add(int(part))
    return sorted([p for p in pages if 1 <= p <= total_pages])

def process_pdf(pdf_file, page_string, custom_name):
    if pdf_file is None: return None, "PDF 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        reader = PdfReader(pdf_file.name)
        total_pages = len(reader.pages)
        target_pages = parse_page_string(page_string, total_pages)
        if not target_pages: return None, f"유효하지 않은 입력입니다. (총 {total_pages}페이지)", gr.update(interactive=False)

        writer = PdfWriter()
        for p in target_pages: writer.add_page(reader.pages[p - 1])

        original_name = os.path.basename(pdf_file.name)
        name_only, ext = os.path.splitext(original_name)

        if custom_name and custom_name.strip():
            output_path = f"{custom_name.strip()}{ext}"
        else:
            safe_page_str = page_string.replace(' ', '').replace(',', '_')
            output_path = f"{name_only}_pages_{safe_page_str}{ext}"

        with open(output_path, "wb") as f: writer.write(f)
        return output_path, f"성공적으로 추출되었습니다! (페이지: {target_pages})", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류 발생: {str(e)}", gr.update(interactive=False)

# ==========================================
# 2. 오디오 분할 (Audio Swordmaster)
# ==========================================
def parse_time_to_ms(time_str):
    if not time_str or str(time_str).strip() in ["", "0"]: return 0
    time_str = str(time_str).strip()
    try:
        if ':' in time_str:
            parts = time_str.split(':')
            if len(parts) == 2: return int((int(parts[0]) * 60 + float(parts[1])) * 1000)
            elif len(parts) == 3: return int((int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])) * 1000)
        else: return int(float(time_str) * 1000)
    except Exception: raise ValueError(f"잘못된 시간 형식: {time_str}")
    return 0

def process_audio(audio_file, start_time_str, end_time_str, remove_silence, custom_name):
    if audio_file is None: return None, "오디오 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        audio = AudioSegment.from_file(audio_file)
        start_ms = parse_time_to_ms(start_time_str)
        end_ms = len(audio) if not end_time_str or str(end_time_str).strip() in ["", "0"] else parse_time_to_ms(end_time_str)

        if start_ms >= end_ms and end_ms != len(audio): return None, "시작이 종료보다 클 수 없습니다.", gr.update(interactive=False)
        audio = audio[start_ms:end_ms]

        if remove_silence:
            chunks = split_on_silence(audio, min_silence_len=500, silence_thresh=-40)
            if chunks: audio = sum(chunks)

        original_name = os.path.basename(audio_file)
        name_only, ext = os.path.splitext(original_name)

        ext_lower = ext.lower()
        if ext_lower not in ['.mp3', '.m4a', '.wav', '.ogg', '.flac']:
            ext_lower = '.mp3'
        export_format = "ipod" if ext_lower == ".m4a" else ext_lower.replace('.', '')

        if custom_name and custom_name.strip():
            output_path = f"{custom_name.strip()}{ext_lower}"
        else:
            silence_tag = "_trimmed" if remove_silence else ""
            output_path = f"{name_only}_{start_ms//1000}s_to_{end_ms//1000}s{silence_tag}{ext_lower}"

        audio.export(output_path, format=export_format)

        try:
            orig_meta = mutagen.File(audio_file)
            if orig_meta and orig_meta.tags:
                new_meta = mutagen.File(output_path)
                new_meta.tags = orig_meta.tags
                new_meta.save()
        except Exception as e:
            pass

        return output_path, "성공적으로 처리되었습니다!", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류 발생: {str(e)}", gr.update(interactive=False)

# ==========================================
# 3. 오디오 N등분 (Audio N-Splitter)
# ==========================================
def process_audio_nsplit(audio_file, n_splits, custom_name):
    if audio_file is None: return None, "오디오 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        n = int(n_splits)
        if n < 2: return None, "2 이상의 숫자를 입력해주세요.", gr.update(interactive=False)

        audio = AudioSegment.from_file(audio_file)
        total_ms = len(audio)
        segment_ms = total_ms // n

        original_name = os.path.basename(audio_file)
        name_only, ext = os.path.splitext(original_name)
        ext_lower = ext.lower()
        if ext_lower not in ['.mp3', '.m4a', '.wav', '.ogg', '.flac']:
            ext_lower = '.mp3'
        export_format = "ipod" if ext_lower == ".m4a" else ext_lower.replace('.', '')

        base_name = custom_name.strip() if custom_name and custom_name.strip() else name_only

        output_files = []
        for i in range(n):
            start = i * segment_ms
            end = (i + 1) * segment_ms if i < n - 1 else total_ms
            chunk = audio[start:end]

            chunk_name = f"{base_name}_part{i+1}{ext_lower}"
            chunk.export(chunk_name, format=export_format)

            try:
                orig_meta = mutagen.File(audio_file)
                if orig_meta and orig_meta.tags:
                    new_meta = mutagen.File(chunk_name)
                    new_meta.tags = orig_meta.tags
                    new_meta.save()
            except Exception as e:
                pass

            output_files.append(chunk_name)

        zip_filename = f"{base_name}_{n}splits.zip"
        with zipfile.ZipFile(zip_filename, 'w') as zipf:
            for f in output_files:
                zipf.write(f, os.path.basename(f))

        return output_files, f"성공적으로 {n}등분 되었습니다!", gr.update(value=zip_filename, interactive=True)
    except Exception as e:
        return None, f"오류 발생: {str(e)}", gr.update(interactive=False)

# ==========================================
# 4. PDF 병합 및 해제 (File Fusion)
# ==========================================
def process_fusion(pdf_files, password, custom_name):
    if not pdf_files: return None, "합칠 PDF 파일들을 업로드해주세요.", gr.update(interactive=False)
    try:
        writer = PdfWriter()
        for pdf in pdf_files:
            reader = PdfReader(pdf.name)
            if reader.is_encrypted:
                if password:
                    reader.decrypt(password)
                else:
                    return None, f"잠긴 파일이 포함되어 있습니다. 비밀번호를 입력해주세요.", gr.update(interactive=False)

            for page in reader.pages:
                writer.add_page(page)

        original_name = os.path.basename(pdf_files[0].name)
        name_only, ext = os.path.splitext(original_name)

        if custom_name and custom_name.strip():
            output_path = f"{custom_name.strip()}.pdf"
        else:
            others_count = len(pdf_files) - 1
            count_tag = f"_and_{others_count}others" if others_count > 0 else ""
            output_path = f"{name_only}{count_tag}_fusion.pdf"

        with open(output_path, "wb") as f:
            writer.write(f)

        return output_path, f"총 {len(pdf_files)}개의 파일이 성공적으로 병합(해제)되었습니다!", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류 발생: {str(e)}", gr.update(interactive=False)

# ==========================================
# 5. 영상 음원 추출 (Audio Extractor)
# ==========================================
def process_video_extractor(video_file, custom_name):
    if video_file is None: return None, "비디오 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        # 비디오 파일 불러오기
        clip = VideoFileClip(video_file.name)

        # 파일명 생성
        original_name = os.path.basename(video_file.name)
        name_only, ext = os.path.splitext(original_name)

        if custom_name and custom_name.strip():
            output_path = f"{custom_name.strip()}.mp3"
        else:
            output_path = f"{name_only}_extracted_audio.mp3"

        # 오디오만 따로 저장 (logger=None으로 터미널 로그 숨김)
        clip.audio.write_audiofile(output_path, logger=None)
        clip.close()

        return output_path, "성공적으로 영상에서 소리만 추출되었습니다!", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류 발생: {str(e)}", gr.update(interactive=False)

# ==========================================
# 6. PDF 잠금 및 워터마크 (PDF Security)
# ==========================================
def process_pdf_security(pdf_file, new_password, watermark_text, custom_name):
    if pdf_file is None: return None, "PDF 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        reader = PdfReader(pdf_file.name)
        writer = PdfWriter()

        # 텍스트 워터마크 생성 로직 (입력값이 있을 때만)
        watermark_reader = None
        if watermark_text and watermark_text.strip():
            packet = io.BytesIO()
            # A4 기준 중앙에 워터마크 그리기
            can = canvas.Canvas(packet, pagesize=A4)
            can.setFont("HYGothic-Medium", 60) # 한글 지원 폰트로 변경
            can.setFillColorRGB(0.5, 0.5, 0.5, alpha=0.3) # 반투명한 회색

            # 페이지 중앙으로 이동 후 대각선으로 회전하여 텍스트 삽입
            can.translate(300, 400)
            can.rotate(45)
            can.drawCentredString(0, 0, watermark_text)
            can.save()

            packet.seek(0)
            watermark_reader = PdfReader(packet)
            watermark_page = watermark_reader.pages[0]

        # 모든 페이지를 돌며 합치기
        for page in reader.pages:
            if watermark_reader:
                page.merge_page(watermark_page) # 워터마크 도장 찍기
            writer.add_page(page)

        # 비밀번호를 입력했다면 암호 걸기
        if new_password and new_password.strip():
            writer.encrypt(new_password)

        # 파일명 생성
        original_name = os.path.basename(pdf_file.name)
        name_only, ext = os.path.splitext(original_name)

        if custom_name and custom_name.strip():
            output_path = f"{custom_name.strip()}.pdf"
        else:
            output_path = f"{name_only}_secured.pdf"

        with open(output_path, "wb") as f:
            writer.write(f)

        return output_path, "성공적으로 보안 처리가 완료되었습니다!", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류 발생: {str(e)}", gr.update(interactive=False)

# ==========================================
# 7. Web UI 구성
# ==========================================
custom_theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="blue",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Pretendard"), "Arial", "sans-serif"]
).set(
    block_radius="lg",
    block_border_width="1px",
    block_shadow="*shadow_sm",
    button_primary_background_fill="*primary_500",
    button_primary_background_fill_hover="*primary_600"
)

with gr.Blocks(theme=custom_theme, title="File Swordmaster") as demo:
    gr.HTML("""
    <div style="text-align: center; max-width: 800px; margin: 20px auto; padding-bottom: 10px; border-bottom: 2px solid #e2e8f0;">
        <h1 style="font-weight: 800; font-size: 2.8rem; margin-bottom: 10px; color: #4338ca; letter-spacing: -1px;">File Swordmaster</h1>
        <p style="font-size: 1.1rem; color: #64748b; margin-top: 0;">개 싹싹 파일 소드마스터</p>
    </div>
    """)

    with gr.Tab("PDF 분할"):
        gr.HTML("<h3 style='color: #334155; margin-bottom: 15px;'>원하는 페이지만 정밀하게 추출합니다.</h3>")
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    pdf_input = gr.File(label="PDF 업로드", file_types=[".pdf"])
                    pdf_pages = gr.Textbox(label="추출 페이지 (예: 1, 3, 5-10)")
                    pdf_custom_name = gr.Textbox(label="저장할 파일명 (선택, 확장자 생략)", placeholder="입력하지 않으면 자동 생성됩니다")
                    pdf_btn = gr.Button("자르기 실행", variant="primary", size="lg")
            with gr.Column(scale=1):
                with gr.Group():
                    pdf_out_file = gr.File(label="결과물")
                    pdf_out_msg = gr.Textbox(label="상태 메시지", interactive=False)
                    # 균형을 맞추기 위한 빈 공간 확보
                    gr.HTML("<br><br>")
                    pdf_download_btn = gr.DownloadButton("결과물 다운로드", interactive=False, variant="secondary")
        pdf_btn.click(process_pdf, inputs=[pdf_input, pdf_pages, pdf_custom_name], outputs=[pdf_out_file, pdf_out_msg, pdf_download_btn])

    with gr.Tab("오디오 분할"):
        gr.HTML("<h3 style='color: #334155; margin-bottom: 15px;'>정밀 타임라인 컷팅 및 무음 구간을 제거합니다. 앨범 표지 등과 같은 메타데이터는 유지됩니다.</h3>")
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    audio_input = gr.Audio(label="오디오 업로드", type="filepath")
                    with gr.Row():
                        audio_start = gr.Textbox(label="시작 시간 (초/분:초)", value="0")
                        audio_end = gr.Textbox(label="종료 시간 (0이면 끝까지)", value="0")
                    # 레이아웃 정렬을 위해 Checkbox와 Textbox를 Row로 묶음
                    with gr.Row():
                        audio_sil = gr.Checkbox(label="무음 구간 자동 제거", value=False)
                        audio_custom_name = gr.Textbox(label="저장할 파일명 (선택, 확장자 생략)", placeholder="입력하지 않으면 자동 생성됩니다")
                    audio_btn = gr.Button("자르기 실행", variant="primary", size="lg")
            with gr.Column(scale=1):
                with gr.Group():
                    audio_out_file = gr.File(label="결과물")
                    audio_out_msg = gr.Textbox(label="상태 메시지", interactive=False)
                    # 균형을 맞추기 위한 빈 공간 확보
                    gr.HTML("<br><br><br>")
                    audio_download_btn = gr.DownloadButton("결과물 다운로드", interactive=False, variant="secondary")
        audio_btn.click(process_audio, inputs=[audio_input, audio_start, audio_end, audio_sil, audio_custom_name], outputs=[audio_out_file, audio_out_msg, audio_download_btn])

    with gr.Tab("오디오 N등분"):
        gr.HTML("<h3 style='color: #334155; margin-bottom: 15px;'>긴 오디오 파일을 원하는 개수로 균등하게 분할합니다.</h3>")
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    nsplit_input = gr.Audio(label="오디오 업로드", type="filepath")
                    with gr.Row():
                        nsplit_count = gr.Number(label="나눌 개수 (N)", value=2, minimum=2, precision=0)
                        nsplit_custom_name = gr.Textbox(label="저장할 파일명 (선택)", placeholder="자동 생성")
                    nsplit_btn = gr.Button("N등분 실행", variant="primary", size="lg")
            with gr.Column(scale=1):
                with gr.Group():
                    nsplit_out_file = gr.File(label="개별 파일 확인", file_count="multiple")
                    nsplit_out_msg = gr.Textbox(label="상태 메시지", interactive=False)
                    gr.HTML("<br>")
                    nsplit_download_btn = gr.DownloadButton("전체 ZIP 다운로드", interactive=False, variant="secondary")
        nsplit_btn.click(process_audio_nsplit, inputs=[nsplit_input, nsplit_count, nsplit_custom_name], outputs=[nsplit_out_file, nsplit_out_msg, nsplit_download_btn])

    with gr.Tab("PDF 병합 및 해제"):
        gr.HTML("<h3 style='color: #334155; margin-bottom: 15px;'>여러 문서를 하나로 합치거나 암호화된 문서의 잠금을 해제합니다.</h3>")
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    fusion_input = gr.File(label="여러 PDF 파일 업로드 (순서대로 병합됨)", file_types=[".pdf"], file_count="multiple")
                    with gr.Row():
                        fusion_pw = gr.Textbox(label="비밀번호 (잠긴 파일이 있을 경우 입력)", type="password")
                        fusion_custom_name = gr.Textbox(label="저장할 파일명 (선택)", placeholder="자동 생성")
                    fusion_btn = gr.Button("합치기 / 잠금해제", variant="primary", size="lg")
            with gr.Column(scale=1):
                with gr.Group():
                    fusion_out_file = gr.File(label="결과물")
                    fusion_out_msg = gr.Textbox(label="상태 메시지", interactive=False)
                    gr.HTML("<br>")
                    fusion_download_btn = gr.DownloadButton("결과물 다운로드", interactive=False, variant="secondary")
        fusion_btn.click(process_fusion, inputs=[fusion_input, fusion_pw, fusion_custom_name], outputs=[fusion_out_file, fusion_out_msg, fusion_download_btn])

    with gr.Tab("영상 음원 추출"):
        gr.HTML("<h3 style='color: #334155; margin-bottom: 15px;'>비디오 파일에서 오디오 트랙만 추출합니다.</h3>")
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    video_input = gr.File(label="영상 파일 업로드")
                    video_custom_name = gr.Textbox(label="저장할 파일명 (선택, 확장자 생략)", placeholder="입력하지 않으면 자동 생성됩니다")
                    video_btn = gr.Button("음원 추출 실행", variant="primary", size="lg")
            with gr.Column(scale=1):
                with gr.Group():
                    video_out_file = gr.File(label="추출된 오디오 (.mp3)")
                    video_out_msg = gr.Textbox(label="상태 메시지", interactive=False)
                    gr.HTML("<br>")
                    video_download_btn = gr.DownloadButton("결과물 다운로드", interactive=False, variant="secondary")
        video_btn.click(process_video_extractor, inputs=[video_input, video_custom_name], outputs=[video_out_file, video_out_msg, video_download_btn])

    with gr.Tab("PDF 보안/워터마크"):
        gr.HTML("<h3 style='color: #334155; margin-bottom: 15px;'>문서에 비밀번호를 설정하거나 투명한 텍스트 워터마크를 각인합니다.</h3>")
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    security_input = gr.File(label="원본 PDF 업로드", file_types=[".pdf"])
                    # 레이아웃 정렬을 위해 텍스트 박스들을 Row로 묶음
                    with gr.Row():
                        security_pw = gr.Textbox(label="새로 설정할 비밀번호 (공란 시 설정 안 함)", type="password")
                        security_wm = gr.Textbox(label="대각선 워터마크 텍스트", placeholder="예: 대외비, 기밀문서")
                    security_custom_name = gr.Textbox(label="저장할 파일명 (선택, 확장자 생략)", placeholder="입력하지 않으면 자동 생성됩니다")
                    security_btn = gr.Button("보안 문서 생성", variant="primary", size="lg")
            with gr.Column(scale=1):
                with gr.Group():
                    security_out_file = gr.File(label="결과물")
                    security_out_msg = gr.Textbox(label="상태 메시지", interactive=False)
                    # 균형을 맞추기 위한 빈 공간 확보
                    gr.HTML("<br><br>")
                    security_download_btn = gr.DownloadButton("결과물 다운로드", interactive=False, variant="secondary")
        security_btn.click(process_pdf_security, inputs=[security_input, security_pw, security_wm, security_custom_name], outputs=[security_out_file, security_out_msg, security_download_btn])

demo.launch()

  with gr.Blocks(theme=custom_theme, title="File Swordmaster") as demo:



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba195322180e528ac3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
